<a href="https://colab.research.google.com/github/Agac-ac/FUNDAI-Laboratories-Agac-ac/blob/main/Lab5_Rule_Based_Expert_System_Agac_ac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Design a Rule-Based Expert System



## Fundamentals of Artificial Intelligence



**Name:** Jhonel L. Agac-ac
**Course:**  BSCS AI
**Section:** 09282-FUNDAI
**Date:** 09/24/26

**Domain:** Medical Symptom Triage Bot
**GitHub URL:** https://github.com/Agac-ac/FUNDAI-Laboratories-Agac-ac

## Disclaimer



This expert system is for educational purposes only.
It is not a real medical diagnostic system.

In [12]:
RULES = [
    {
        "id": "R1",
        "priority": 10,
        "if": {
            "fever": True,
            "cough": True,
            "body_ache": True
        },
        "then": {
            "flu_like": True
        },
        "description": "Fever, cough, and body ache suggest flu-like symptoms."
    },
    {
        "id": "R2",
        "priority": 9,
        "if": {
            "fever": True,
            "sore_throat": True
        },
        "then": {
            "throat_infection_possible": True
        },
        "description": "Fever and sore throat suggest possible throat infection."
    },
    {
        "id": "R3",
        "priority": 9,
        "if": {
            "cough": True,
            "shortness_of_breath": True
        },
        "then": {
            "respiratory_warning": True
        },
        "description": "Cough and shortness of breath require medical attention."
    },
    {
        "id": "R4",
        "priority": 8,
        "if": {
            "flu_like": True
        },
        "then": {
            "suspect_flu": True
        },
        "description": "Flu-like symptoms lead to suspected flu."
    },
    {
        "id": "R5",
        "priority": 7,
        "if": {
            "suspect_flu": True
        },
        "then": {
            "recommend_rest_and_hydration": True
        },
        "description": "Suspected flu recommends rest and hydration."
    },
    {
        "id": "R6",
        "priority": 10,
        "if": {
            "respiratory_warning": True
        },
        "then": {
            "recommend_doctor": True
        },
        "description": "Respiratory warning recommends doctor consultation."
    },
    {
        "id": "R7",
        "priority": 12,
        "if": {
            "severe_chest_pain": True
        },
        "then": {
            "emergency": True
        },
        "description": "Severe chest pain requires emergency care."
    },
    {
        "id": "R8",
        "priority": 6,
        "if": {
            "throat_infection_possible": True
        },
        "then": {
            "recommend_doctor": True
        },
        "description": "Possible throat infection recommends doctor consultation."
    }
]

In [13]:
DEFAULT_FACTS = {
    "fever": False,
    "cough": False,
    "body_ache": False,
    "sore_throat": False,
    "shortness_of_breath": False,
    "severe_chest_pain": False
}

In [14]:
def make_facts(**overrides):
    facts = dict(DEFAULT_FACTS)
    facts.update(overrides)
    return facts


In [15]:
def rule_condition_matches(rule_if, facts):
    for fact_name, required_value in rule_if.items():
        if facts.get(fact_name, False) != required_value:
            return False
    return True

In [16]:
def forward_chain(facts, rules, trace=True):
    facts = dict(facts)
    fired_rules = []

    sorted_rules = sorted(rules, key=lambda rule: rule.get("priority", 0), reverse=True)

    changed = True

    while changed:
        changed = False

        for rule in sorted_rules:
            if rule["id"] in fired_rules:
                continue

            if rule_condition_matches(rule["if"], facts):
                if trace:
                    print(f"Rule {rule['id']} fired.")
                    print(f"Description: {rule.get('description', 'No description')}")
                    print(f"New facts added: {rule['then']}")
                    print("-" * 50)

                for fact_name, fact_value in rule["then"].items():
                    if facts.get(fact_name) != fact_value:
                        facts[fact_name] = fact_value
                        changed = True

                fired_rules.append(rule["id"])

    return facts, fired_rules

In [17]:
def explain_inference(fired_rule_ids, rules):
    rule_map = {rule["id"]: rule for rule in rules}

    print("=" * 60)
    print("EXPLANATION OF INFERENCE")
    print("=" * 60)

    if len(fired_rule_ids) == 0:
        print("No rules were fired.")
        return

    for rule_id in fired_rule_ids:
        rule = rule_map[rule_id]

        print(f"Rule {rule_id}")
        print(f"IF: {rule['if']}")
        print(f"THEN: {rule['then']}")
        print(f"Reason: {rule.get('description', 'No description')}")
        print("-" * 60)


In [18]:
ADVICE_PRIORITY = [
    (
        "emergency",
        "EMERGENCY: Please seek immediate medical help."
    ),
    (
        "recommend_doctor",
        "RECOMMENDATION: Please consult a doctor."
    ),
    (
        "suspect_flu",
        "POSSIBLE CONDITION: Flu-like illness."
    ),
    (
        "throat_infection_possible",
        "POSSIBLE CONDITION: Possible throat infection."
    ),
    (
        "respiratory_warning",
        "WARNING: Respiratory symptoms detected."
    ),
    (
        "recommend_rest_and_hydration",
        "ADVICE: Rest and stay hydrated."
    )
]

def get_conclusions(facts):
    conclusions = []

    for fact_name, message in ADVICE_PRIORITY:
        if facts.get(fact_name, False):
            conclusions.append(message)

    if len(conclusions) == 0:
        conclusions.append(
            "No specific condition matched. "
            "Please provide more symptoms or consult a professional."
        )

    return conclusions

In [19]:
def run_case(case_name, facts):
    print("=" * 60)
    print(f"CASE: {case_name}")
    print("=" * 60)
    print("Input facts:")
    print(facts)
    print("-" * 60)

    final_facts, fired_rules = forward_chain(facts, RULES, trace=True)

    print("Final facts:")
    print(final_facts)

    print("\nConclusions:")
    for conclusion in get_conclusions(final_facts):
        print("-", conclusion)

    print()

    explain_inference(fired_rules, RULES)

    print("\n\n")

In [20]:
# Case 1: Flu-like symptoms
case1 = make_facts(
    fever=True,
    cough=True,
    body_ache=True
)

# Case 2: Respiratory warning
case2 = make_facts(
    cough=True,
    shortness_of_breath=True
)

# Case 3: Emergency condition
case3 = make_facts(
    severe_chest_pain=True
)

# Case 4: No matching symptoms
case4 = make_facts()

run_case("Flu-like symptoms", case1)
run_case("Respiratory warning", case2)
run_case("Emergency chest pain", case3)
run_case("No symptoms", case4)

CASE: Flu-like symptoms
Input facts:
{'fever': True, 'cough': True, 'body_ache': True, 'sore_throat': False, 'shortness_of_breath': False, 'severe_chest_pain': False}
------------------------------------------------------------
Rule R1 fired.
Description: Fever, cough, and body ache suggest flu-like symptoms.
New facts added: {'flu_like': True}
--------------------------------------------------
Rule R4 fired.
Description: Flu-like symptoms lead to suspected flu.
New facts added: {'suspect_flu': True}
--------------------------------------------------
Rule R5 fired.
Description: Suspected flu recommends rest and hydration.
New facts added: {'recommend_rest_and_hydration': True}
--------------------------------------------------
Final facts:
{'fever': True, 'cough': True, 'body_ache': True, 'sore_throat': False, 'shortness_of_breath': False, 'severe_chest_pain': False, 'flu_like': True, 'suspect_flu': True, 'recommend_rest_and_hydration': True}

Conclusions:
- POSSIBLE CONDITION: Flu-lik

In [21]:
def ask_yes_no(question):
    answer = input(question + " (yes/no): ")
    answer = answer.strip().lower()
    return answer in ["yes", "y", "true", "1"]


def collect_facts():
    facts = dict(DEFAULT_FACTS)

    print("Medical Symptom Triage Bot")
    print("Answer the following questions.")
    print("-" * 40)

    facts["fever"] = ask_yes_no("Do you have fever?")
    facts["cough"] = ask_yes_no("Do you have cough?")
    facts["body_ache"] = ask_yes_no("Do you have body ache?")
    facts["sore_throat"] = ask_yes_no("Do you have sore throat?")
    facts["shortness_of_breath"] = ask_yes_no("Do you have shortness of breath?")
    facts["severe_chest_pain"] = ask_yes_no("Do you have severe chest pain?")

    return facts


def interactive_mode():
    facts = collect_facts()

    final_facts, fired_rules = forward_chain(facts, RULES, trace=True)

    print("\nConclusions:")
    for conclusion in get_conclusions(final_facts):
        print("-", conclusion)

    explain_inference(fired_rules, RULES)

In [ ]:
interactive_mode()

Medical Symptom Triage Bot
Answer the following questions.
----------------------------------------


## Knowledge Acquisition



### Domain



Medical symptom triage.

### Source of Knowledge



* General common-sense health rules


* Instructor-approved sample rules


* Basic first-aid recommendations


* Educational examples only



### Rules Collected



1. Fever, cough, and body ache may suggest flu-like symptoms.


2. Cough and shortness of breath require medical consultation.


3. Severe chest pain requires emergency care.


4. Suspected flu recommends rest and hydration.



### Limitations



* This system is not a real medical diagnostic tool.


* The rules are simplified.


* The system does not consider medical history, age, allergies, or severity.


* A real medical expert system requires validation by licensed professionals.

## Guide Questions and Answers

### 1. What is an expert system?

**Answer:** An expert system is a computer program that emulates the decision-making ability of a human expert. It uses a knowledge base of facts and rules within a specific domain to solve complex problems and provide advice or explanations.

### 2. What are the main components of an expert system?

**Answer:** The main components of an expert system are:
*   **Knowledge Base:** Contains facts and heuristic rules related to the problem domain (e.g., the `RULES` and `DEFAULT_FACTS` in this system).
*   **Inference Engine:** Applies the rules in the knowledge base to the facts to deduce new facts or reach conclusions (e.g., the `forward_chain` function).
*   **User Interface:** Allows interaction between the user and the system (e.g., the `collect_facts` and `interactive_mode` functions).
*   **Explanation Facility:** Provides justifications for the conclusions drawn by the system (e.g., the `explain_inference` function).

### 3. What is forward chaining?

**Answer:** Forward chaining is an inference strategy in expert systems that starts with known facts and applies rules to deduce new facts until a goal is reached or no more rules can be applied. It is a data-driven approach, moving from observations to conclusions.

### 4. How does your system use forward chaining?

**Answer:** My system uses forward chaining in the `forward_chain` function. It takes an initial set of facts and iterates through the `RULES`. For each rule, it checks if its `if` conditions match the current facts. If they match, the rule is "fired," and the `then` part of the rule asserts new facts into the system, potentially triggering other rules in subsequent iterations. This process continues until no more rules can be fired.

### 5. Which rules fired in Test Case 1?

**Answer:** In Test Case 1 ("Flu-like symptoms") with `fever=True, cough=True, body_ache=True`, the following rules fired:
*   **R1:** Fever, cough, and body ache suggest flu-like symptoms.
*   **R4:** Flu-like symptoms lead to suspected flu.
*   **R5:** Suspected flu recommends rest and hydration.

### 6. Which rules fired in Test Case 2?

**Answer:** In Test Case 2 ("Respiratory warning") with `cough=True, shortness_of_breath=True`, the following rules fired:
*   **R3:** Cough and shortness of breath require medical attention.
*   **R6:** Respiratory warning recommends doctor consultation.

### 7. Why is an explanation facility important?

**Answer:** An explanation facility is crucial for several reasons:
*   **Transparency:** It allows users to understand how the system arrived at its conclusions, fostering trust.
*   **Debugging:** It helps developers identify errors or gaps in the knowledge base or inference logic.
*   **Education:** It can be used as a teaching tool to explain complex reasoning processes.
*   **Validation:** It provides a basis for human experts to review and validate the system's logic.

### 8. What challenges did you encounter while designing the rules?

**Answer:** Some challenges encountered while designing the rules could include:
*   **Completeness:** Ensuring all relevant scenarios are covered by rules.
*   **Consistency:** Avoiding contradictory rules that might lead to conflicting conclusions.
*   **Prioritization:** Assigning appropriate priorities to rules to ensure the most critical or specific rules fire first (e.g., `priority` attribute).
*   **Granularity:** Deciding the right level of detail for facts and rules.
*   **Ambiguity:** Translating natural language medical concepts into precise logical rules.

### 9. What are the limitations of your expert system?

**Answer:** The limitations of this expert system are explicitly stated in the notebook and include:
*   It is **not a real medical diagnostic tool**.
*   The rules are **simplified**.
*   It does not consider medical history, age, allergies, or severity.
*   A real medical expert system requires validation by licensed professionals.

### 10. Why should this system not be used for real medical diagnosis?

**Answer:** This system should not be used for real medical diagnosis because:
*   It is designed **for educational purposes only** and is a simplified model.
*   It lacks the comprehensive knowledge, nuance, and contextual understanding of a human doctor.
*   It does not account for individual patient history, co-morbidities, medication interactions, or the full spectrum of symptom presentations.
*   Medical diagnosis requires the judgment, experience, and ethical considerations of licensed professionals, which cannot be replicated by a basic rule-based system.

# REFLECTION

This Lab 5 exercise provided a practical understanding of designing and implementing a rule-based expert system. I learned about the core components of such systems, including the knowledge base (rules and facts), the inference engine (forward chaining), and the crucial role of an explanation facility. The process involved defining a clear domain, collecting knowledge, and translating that knowledge into a structured set of 'IF-THEN' rules with priorities.

# CONCLUSION

This Lab 5 successfully demonstrated the implementation of a simplified rule-based expert system for medical symptom triage. The system effectively utilizes forward chaining to process a given set of symptoms (facts) against a predefined set of medical rules to reach conclusions and provide advice. The inclusion of an explanation facility further enhances the system's transparency, allowing users to understand the reasoning behind the recommendations.

While this expert system serves as an excellent educational model, it is crucial to reiterate its limitations. It is not intended for real medical diagnosis due to its simplified rule set, lack of contextual understanding, and inability to account for individual patient complexities. However, it provides a solid foundation for understanding the principles of artificial intelligence in decision-making and problem-solving within specific domains. The exercise underscored the power of symbolic AI for explicit knowledge representation and logical inference, as well as the significant effort required to build reliable and robust knowledge bases for real-world applications.